In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pengenalan primitives

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Beta release dari model eksekusi baru kini telah tersedia. Model directed execution memberikan fleksibilitas lebih dalam menyesuaikan alur kerja error mitigation-mu. Lihat panduan [Directed execution model](/guides/directed-execution-model) untuk informasi lebih lanjut.

<span id="qpu-access-patterns"></span>
## Mengapa Qiskit memperkenalkan primitives?
Mirip dengan masa-masa awal komputer klasik, ketika developer harus memanipulasi register CPU secara langsung, antarmuka awal ke QPU hanya mengembalikan data mentah dari elektronik kontrol.
Hal ini tidak menjadi masalah besar ketika QPU masih berada di laboratorium dan hanya bisa diakses langsung oleh para peneliti.
Menyadari bahwa kebanyakan developer tidak perlu dan tidak harus memahami cara mengolah data mentah tersebut menjadi angka 0 dan 1, Qiskit memperkenalkan `backend.run`, sebuah abstraksi pertama untuk mengakses QPU di cloud. Ini memungkinkan developer untuk bekerja dengan format data yang sudah familiar dan fokus pada gambaran besarnya.

Seiring akses ke QPU semakin meluas, dan dengan semakin banyaknya algoritma kuantum yang dikembangkan,
kebutuhan akan abstraksi tingkat yang lebih tinggi pun muncul kembali. Sebagai respons, Qiskit memperkenalkan
antarmuka primitives, yang dioptimalkan untuk dua tugas inti dalam pengembangan algoritma kuantum:
estimasi nilai ekspektasi (`Estimator`) dan pengambilan sampel Circuit (`Sampler`). Tujuannya adalah sekali lagi
membantu developer untuk lebih fokus pada inovasi dan lebih sedikit pada konversi data. Antarmuka primitives menggantikan antarmuka `backend.run`, karena `Sampler` memberikan akses hardware langsung yang sama seperti yang ditawarkan oleh `backend.run`.

## Apa itu primitive?
Sistem komputasi dibangun di atas beberapa lapisan abstraksi. Abstraksi memungkinkan kamu untuk fokus pada
tingkat detail tertentu yang relevan dengan tugas yang sedang dikerjakan. Semakin dekat kamu dengan hardware,
semakin rendah tingkat abstraksi yang kamu butuhkan (misalnya, kamu mungkin perlu memindahkan atau memanipulasi data di tingkat instruksi CPU). Semakin kompleks tugas yang ingin kamu lakukan,
semakin tinggi tingkat abstraksinya (misalnya, kamu bisa menggunakan library pemrograman untuk melakukan
perhitungan aljabar).

Dalam konteks ini, sebuah *primitive* adalah instruksi pemrosesan terkecil, blok bangunan paling sederhana yang darinya
seseorang bisa membuat sesuatu yang berguna untuk tingkat abstraksi tertentu.

Kemajuan terbaru dalam komputasi kuantum telah meningkatkan kebutuhan untuk bekerja pada tingkat abstraksi yang lebih tinggi.
Seiring bidang ini bergerak menuju unit pemrosesan kuantum (QPU) yang lebih besar dan alur kerja yang lebih kompleks, fokusnya bergeser dari berinteraksi dengan sinyal Qubit individual ke melihat perangkat kuantum sebagai sistem yang menjalankan tugas-tugas yang diperlukan.

Dua tugas yang paling umum untuk komputer kuantum adalah pengambilan sampel keadaan kuantum dan penghitungan nilai ekspektasi.
Tugas-tugas inilah yang mendorong desain Qiskit primitives: **Estimator** dan **Sampler**.

- Estimator menghitung nilai ekspektasi dari observable terhadap keadaan yang disiapkan oleh Circuit kuantum.
- Sampler mengambil sampel register output dari eksekusi Circuit kuantum.

Singkatnya, model komputasi yang diperkenalkan oleh Qiskit primitives membawa pemrograman kuantum selangkah lebih dekat
ke tempat pemrograman klasik berada saat ini, di mana fokusnya lebih sedikit pada detail hardware dan lebih banyak pada hasil
yang ingin kamu capai.

## Definisi dan implementasi primitive
Ada dua jenis Qiskit primitives: kelas dasar, dan implementasinya. Qiskit primitives didefinisikan oleh kelas dasar primitive open-source yang ada di Qiskit SDK (di modul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Provider (seperti Qiskit Runtime) dapat menggunakan kelas dasar ini untuk menurunkan implementasi Sampler dan Estimator mereka sendiri. Sebagian besar pengguna akan berinteraksi dengan implementasi provider, bukan primitive dasar.

### Kelas dasar
[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) dan [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Kelas dasar abstrak yang mendefinisikan antarmuka umum untuk mengimplementasikan primitives. Semua kelas lain di modul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) mewarisi dari kelas dasar ini. Developer harus menggunakan kelas ini jika mereka tertarik untuk membuat model eksekusi berbasis primitives mereka sendiri untuk provider tertentu. Kelas-kelas ini mungkin juga berguna bagi mereka yang ingin melakukan pemrosesan yang sangat dikustomisasi dan menemukan bahwa implementasi primitives yang sudah ada terlalu sederhana untuk kebutuhan mereka. Pengguna umum tidak akan langsung menggunakan kelas dasar ini.

<span id="implementations"></span>
### Implementasi
Berikut adalah implementasi dari kelas dasar primitives:

- Qiskit Runtime primitives ([`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) dan [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)) menyediakan implementasi yang lebih canggih (misalnya, dengan menyertakan error mitigation) sebagai layanan berbasis cloud. Implementasi primitive dasar ini digunakan untuk mengakses hardware IBM Quantum&reg;. Mereka diakses melalui IBM Qiskit Runtime.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) dan [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Implementasi referensi dari primitives yang menggunakan simulator bawaan Qiskit. Mereka dibangun dengan modul Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information), menghasilkan hasil berdasarkan simulasi statevector ideal. Mereka diakses melalui Qiskit.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) dan [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Kamu bisa menggunakan kelas-kelas ini untuk "membungkus" sumber daya komputasi kuantum apa pun menjadi sebuah primitive. Ini memungkinkan kamu menulis kode bergaya primitive untuk provider yang belum memiliki antarmuka berbasis primitives. Kelas-kelas ini bisa digunakan seperti Sampler dan Estimator biasa, kecuali mereka harus diinisialisasi dengan argumen `backend` tambahan untuk memilih komputer kuantum mana yang akan dijalankan. Mereka diakses menggunakan Qiskit.

## Manfaat Qiskit primitives
Dengan primitives, pengguna Qiskit bisa menulis kode kuantum untuk QPU tertentu tanpa harus secara eksplisit
mengelola setiap detail. Juga, karena adanya lapisan abstraksi tambahan, kamu mungkin bisa lebih mudah
mengakses kemampuan hardware canggih dari provider tertentu. Misalnya, dengan Qiskit Runtime primitives,
kamu bisa memanfaatkan kemajuan terbaru dalam error mitigation dan suppression dengan mengaktifkan opsi seperti [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) pada primitive, daripada membangun implementasi teknik-teknik ini sendiri.

Untuk provider hardware, mengimplementasikan primitives secara native berarti kamu bisa memberikan pengguna cara yang lebih "siap pakai"
untuk mengakses fitur hardware-mu seperti teknik pasca-pemrosesan canggih. Dengan begitu, lebih mudah bagi penggunamu untuk mendapatkan manfaat dari kemampuan terbaik hardware-mu.

## Detail primitive
Seperti yang dijelaskan sebelumnya, semua primitives dibuat dari kelas dasar; oleh karena itu, mereka memiliki struktur dan penggunaan umum yang sama. Misalnya, format input untuk semua Estimator primitives adalah sama. Namun, ada perbedaan dalam implementasi yang membuat masing-masing unik.

> **Note:** Karena sebagian besar pengguna mengakses Qiskit Runtime primitives, contoh-contoh di sisa bagian ini didasarkan pada Qiskit Runtime primitives.

<span id="estimator"></span>
### Estimator
Primitive Estimator menghitung nilai ekspektasi untuk satu atau lebih observable terhadap keadaan yang disiapkan oleh Circuit kuantum. Circuit bisa diparametrisasi, selama nilai parameter juga diberikan sebagai input ke primitive.

Inputnya adalah array dari [PUB.](/guides/primitive-input-output#pubs) Setiap PUB dalam format:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

di mana `parameter values` opsional bisa berupa list atau parameter tunggal. Implementasi Estimator yang berbeda mendukung berbagai opsi konfigurasi. Jika input mengandung pengukuran, pengukuran tersebut diabaikan.

Outputnya adalah [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) yang berisi nilai ekspektasi yang dihitung per pasangan, dan standard error-nya, dalam bentuk `PubResult`. Setiap `PubResult` berisi data dan metadata.

Estimator menggabungkan elemen dari observable dan nilai parameter dengan mengikuti aturan broadcasting NumPy seperti yang dijelaskan di topik [Primitive inputs and outputs](primitive-input-output#broadcasting-rules).

Contoh:

In [1]:
# This cell is hidden from users, it creates the circuits and observables to run

from qiskit_ibm_runtime import EstimatorV2, SamplerV2, QiskitRuntimeService
from qiskit.circuit.random import random_circuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

service = QiskitRuntimeService()
backend = service.least_busy()
phi = Parameter("phi")

circuit1 = random_circuit(10, 5, seed=12345)
circuit1.rzz(phi, 1, 2)
observable1 = SparsePauliOp.from_sparse_list(
    [("ZXYZ", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values1 = np.random.uniform(size=5).T

circuit2 = random_circuit(10, 5, seed=12345)
circuit2.rzz(phi, 1, 2)
observable2 = SparsePauliOp.from_sparse_list(
    [("XZYX", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values2 = np.random.uniform(size=5).T

shots1 = 164
shots2 = 1024

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
circuit1 = pm.run(circuit1)
circuit2 = pm.run(circuit2)
observable1 = observable1.apply_layout(circuit1.layout)
observable2 = observable2.apply_layout(circuit2.layout)

In [2]:
estimator = EstimatorV2(mode=backend)
estimator_job = estimator.run(
    [
        (circuit1, observable1, param_values1),
        (circuit2, observable2, param_values2),
    ]
)

<span id="sampler"></span>
### Sampler
Tugas inti Sampler adalah mengambil sampel register output dari eksekusi satu atau lebih Circuit kuantum. Circuit input bisa diparametrisasi, selama nilai parameter juga diberikan sebagai input ke primitive.

Inputnya adalah satu atau lebih [PUB,](/guides/primitive-input-output#pubs) dalam format:

(`<single circuit>`, `<one or more optional parameter value>`, `<optional shots>`),

di mana bisa ada beberapa item `parameter values`, dan setiap item bisa berupa array atau parameter tunggal, tergantung pada Circuit yang dipilih. Selain itu, input harus mengandung pengukuran.

Outputnya adalah jumlah hitungan atau pengukuran per-shot, sebagai objek [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), tanpa bobot. Namun, kelas result memiliki metode untuk mengembalikan sampel berbobot, seperti hitungan. Lihat [Primitive inputs and outputs](primitive-input-output#broadcasting-rules) untuk detail lengkapnya.

Contoh:

In [3]:
# This cell is hidden from users, add measurement instructions to circuits
circuit1.measure_active()
circuit2.measure_active()

In [4]:
sampler = SamplerV2(mode=backend)
sampler_job = sampler.run(
    [
        (circuit1, param_values1, shots1),
        (circuit2, param_values2, shots2),
    ]
)

## Langkah selanjutnya
> **Tip:** - Baca [Memulai dengan primitives](get-started-with-primitives) untuk mengimplementasikan primitives dalam pekerjaanmu.
>     - Tinjau [contoh-contoh primitives](primitives-examples) yang detail.
>     - Berlatih dengan primitives melalui [pelajaran Cost function](/learning/courses/variational-algorithm-design/cost-functions) di IBM Quantum Learning.
>     - Lihat [referensi API EstimatorV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) dan [referensi API SamplerV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2).
>     - Baca [Migrasi ke V2 primitives](/guides/v2-primitives).